In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
import numpy as np

# 固定随机种子，让 DataLoader 的 shuffle、模型初始化都可复现
# 不固定的话，每次重跑训练曲线会略有差异，对比时容易混淆
torch.manual_seed(42)

# matplotlib 中文字体（按系统取一个能用的）
plt.rcParams['font.sans-serif'] = ['Heiti TC', 'PingFang SC', 'Songti SC', 'STHeiti', 'Arial Unicode MS']
plt.rcParams['axes.unicode_minus'] = False

# 数据预处理: 转 tensor + 归一化
# - ToTensor: 把 PIL 图 (uint8, 0-255) 转成 float tensor, 范围 [0, 1], shape 由 (H, W, C) 变成 (C, H, W)
# - Normalize: 减均值除标准差, 让像素分布零均值、单位方差
#   0.1307 / 0.3081 是 MNIST 训练集像素的均值/标准差 (业界写死的常数)
#   为什么要这一步? 后面归一化那节会展开讲—简单说, 能让训练更稳、收敛更快
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,)),
])

# 第一次跑会自动从网上下载到 ./data/, 之后走本地缓存
train_set = torchvision.datasets.MNIST(root='./data', train=True, download=True, transform=transform)
test_set  = torchvision.datasets.MNIST(root='./data', train=False, download=True, transform=transform)

# DataLoader 把数据集切成 batch
# - 训练集 shuffle=True: 每个 epoch 打乱顺序, 避免模型记住数据出现的次序
# - 测试集 shuffle=False: 评估时顺序无所谓, 关掉 shuffle 让结果可复现
train_loader = DataLoader(train_set, batch_size=128, shuffle=True)
test_loader  = DataLoader(test_set,  batch_size=256, shuffle=False)

print(f'训练集: {len(train_set)} 张')
print(f'测试集: {len(test_set)} 张')
print(f'每张图: {train_set[0][0].shape} (通道 x 高 x 宽) ')

训练集: 60000 张
测试集: 10000 张
每张图: torch.Size([1, 28, 28]) (通道 x 高 x 宽) 


In [ ]:
# 取 9 张样本看一下
fig, axes = plt.subplots(3, 3, figsize=(6, 6))
for i, ax in enumerate(axes.flat):
    img, label = train_set[i]
    # img 已经归一化过, 反归一化回去显示 (不然颜色不对)
    ax.imshow(img.squeeze() * 0.3081 + 0.1307, cmap='gray')
    ax.set_title(f'label: {label}', fontsize=12)
    ax.axis('off')  
plt.suptitle('MNIST 训练集前 9 张', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

In [2]:
# ========================================================
# 手动初始化 4 个参数张量: W1, b1, W2, b2
# ========================================================
# 单独重跑这一段时，再固定一次种子，让权重初始化可复现
torch.manual_seed(42)

# requires_grad=True: 告诉 autograd" 这个张量是参数，需要记录梯度"
# 没有这个标记，loss.backward() 算不出 p.grad
#
# (2.0 / n_in) ** 0.5 是 He 初始化的标准差，专门给 ReLU 网络用
# 经验值，先留着；为什么是 sqrt(2/n_in)，参数初始化那节再讲
W1 = (torch.randn(784, 128) * (2.0 / 784) ** 0.5).requires_grad_()
b1 = torch.zeros(128, requires_grad=True)

# 第 2 层：128 -> 10 (输出 10 个类别的 logits)
W2 = (torch.randn(128, 10) * (2.0 / 128) ** 0.5).requires_grad_()
b2 = torch.zeros(10, requires_grad=True)

# 把 4 个参数放一个列表里, 后面统一更新
params = [W1, b1, W2, b2]
lr = 0.1  # 学习率: 每次梯度下降迈多大一步. 先拍0.1, 学习率与调度那节在细聊

print(f"网络共 {sum(p.numel() for p in params):,} 个参数")

网络共 101,770 个参数
